# Homework: Computational Complexity

## Problem 1: Complexity Analysis

For each of the following code snippets, determine the time complexity and space complexity in Big-O notation. Provide a brief justification for each.

**(a)**

In [1]:
def func_a(data):
    n = len(data)
    total = 0
    for i in range(0, n, 2):
        for j in range(5):
            total += data[i]
    return total

Time: $O(n)$, Space: $O(1)$

time: outer loop is over $n$ and inner loop has constant number of iterations. space: extra variables are scalars (`i`, `j`, `total`).

**(b)**

In [2]:
def func_b(matrix):
    n = len(matrix)
    result = []
    for i in range(n):
        for j in range(i):
            result.append(matrix[i][j])
    return result

Time: $O(n^2)$, Space: $O(n^2)$.

Time: outer loop involves $n$ iterations, inner loop involves $n$ iterations at most.
Space: stores approximately half of the matrix values.

**(c)**

In [3]:
def func_c(n):
    if n <= 1:
        return 1
    return func_c(n // 2) + func_c(n // 2)

Time: $O(n)$, Space: $O(\log n)$.

Recursion. Space complexity is $\log_2(n)$.

**(d)**

In [4]:
def func_d(items):
    seen = set()
    duplicates = []
    for item in items:
        if item in seen:
            duplicates.append(item)
        seen.add(item)
    return duplicates

Time: $O(n)$, Space: $O(n)$.

## Problem 2: Choosing the Right Data Structure

Finding elements common to two collections is a frequent operation in data processing (for example, finding shared gene IDs across two experiments).

**(a)** The following function finds common elements using a naive approach. What is its time complexity if `list1` has n elements and `list2` has m elements? Explain why.

In [5]:
def find_common_naive(list1, list2):
    common = []
    for x in list1:
        for y in list2:
            if x == y and x not in common:
                common.append(x)
    return common

Time complexity is $O(mn)$ since it compares the elements between two lists using nested loop.

**(b)** Write an efficient version called `find_common_fast` that uses a set to achieve better time complexity. What is the time complexity of your version? For example, `find_common_fast([1, 2, 3, 4], [3, 4, 5, 6])` should return `[3, 4]` (in any order).

In [6]:
def find_common_fast(list1, list2):
    set2 = set(list2)   
    common = set()      
    for x in list1:     
        if x in set2:   
            common.add(x)
    return list(common) 

Time complexity is $O(n+m)$.

**(c)** Verify the speedup empirically. Write a script that times both functions on lists of size n = 1000, 5000, 10000, and 20000 (where elements are random integers from 0 to n). Print the time for each function at each size.

In [7]:
import random
import time


def time_function(func, list1, list2):
    start = time.perf_counter()
    func(list1, list2)
    end = time.perf_counter()
    return end - start

random.seed(735)
sizes = [1000, 5000, 10000, 20000]

print("n\tnaive_seconds\tfast_seconds")
for n in sizes:
    list1 = [random.randint(0, n) for _ in range(n)]
    list2 = [random.randint(0, n) for _ in range(n)]

    naive_time = time_function(find_common_naive, list1, list2)
    fast_time = time_function(find_common_fast, list1, list2)

    print(f"{n}\t{naive_time:.6f}\t{fast_time:.6f}")

n	naive_seconds	fast_seconds
1000	0.014605	0.000089
5000	0.364323	0.000450
10000	1.459598	0.000924
20000	6.695912	0.002680


## Problem 3: Empirical Complexity Measurement

The following function performs a computation on a list:

In [8]:
def mystery_function(data):
    n = len(data)
    data = sorted(data)
    total = 0
    for i in range(n):
        left, right = 0, n - 1
        while left < right:
            if data[left] + data[right] == data[i]:
                total += 1
            if data[left] + data[right] < data[i]:
                left += 1
            else:
                right -= 1
    return total

Your task is to empirically determine the time complexity of `mystery_function` using the log-log method from the lecture.

**(a)** Write a timing script that measures the runtime of `mystery_function` for input sizes n = 500, 1000, 2000, 4000, and 8000. Use random integer data for each size. Run each size at least 3 times and take the average.

**(b)** Compute the log-log slope using `scipy.stats.linregress` on the log of the sizes and the log of the times. Report the slope value.

**(c)** Based on the slope, what is the time complexity of `mystery_function`? Explain why this matches (or doesn't match) what you would expect from reading the code.

In [9]:
import math
import random
import time

from scipy.stats import linregress


def average_runtime(n, trials=3):
    runtimes = []
    for _ in range(trials):
        data = [random.randint(0, n) for _ in range(n)]
        start = time.perf_counter()
        mystery_function(data)
        end = time.perf_counter()
        runtimes.append(end - start)
    return sum(runtimes) / len(runtimes), runtimes



random.seed(42)
sizes = [500, 1000, 2000, 4000, 8000]
avg_times = []

print("n\ttrial1\ttrial2\ttrial3\tavg_seconds")
for n in sizes:
    avg_time, trials = average_runtime(n, trials=3)
    avg_times.append(avg_time)
    print(
        f"{n}\t{trials[0]:.6f}\t{trials[1]:.6f}\t{trials[2]:.6f}\t{avg_time:.6f}"
    )

log_n = [math.log(x) for x in sizes]
log_t = [math.log(t) for t in avg_times]
result = linregress(log_n, log_t)

print(f"\nlog-log slope = {result.slope:.6f}")
print(f"intercept = {result.intercept:.6f}")
print(f"rvalue = {result.rvalue:.6f}")

/Users/wenbinwu/miniforge3/lib/python3.9/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


n	trial1	trial2	trial3	avg_seconds
500	0.036713	0.037275	0.036117	0.036702
1000	0.152637	0.158252	0.140840	0.150576
2000	0.621888	0.610786	0.573197	0.601957
4000	2.361691	2.303138	2.303461	2.322763
8000	9.893133	10.367547	9.873698	10.044793

log-log slope = 2.014004
intercept = -15.819443
rvalue = 0.999932


## Problem 4: Improving a Statistical Computation

In Bayesian statistics and spatial statistics, you often need to solve many linear systems with the same coefficient matrix but different right-hand side vectors. For example, drawing samples from a multivariate normal or computing conditional distributions.

The following function solves m linear systems using a loop:

In [10]:
import numpy as np


def solve_systems_naive(A, B):
    """Solve A @ X = B for X, where B has m columns.

    Parameters
    ----------
    A : np.ndarray
        Symmetric positive definite matrix of shape (n, n).
    B : np.ndarray
        Matrix of shape (n, m), each column is a right-hand side vector.

    Returns
    -------
    np.ndarray
        Solution matrix X of shape (n, m).
    """
    n, m = B.shape
    X = np.zeros_like(B)
    for i in range(m):
        X[:, i] = np.linalg.solve(A, B[:, i])
    return X

**(a)** What is the time complexity of `solve_systems_naive` in terms of n and m? Explain what happens inside `np.linalg.solve` on each iteration.

The time complexity is $O(m n^3)$ because the factorization of `A` is $O(n^3)$ over $m$ iterations.

**(b)** Rewrite the function as `solve_systems_cholesky` using `scipy.linalg.cholesky` and `scipy.linalg.cho_solve` to factor A once and then solve each system cheaply. What is the new time complexity? For example, `solve_systems_cholesky(np.array([[2, 1], [1, 2]]), np.array([[1, 0], [0, 1]]))` should return `np.array([[2/3, -1/3], [-1/3, 2/3]])` (the inverse of A, since B is the identity).

In [11]:
import numpy as np
import scipy.linalg


def solve_systems_cholesky(A, B):
    """Solve A @ X = B for X using one Cholesky factorization of A.

    Parameters
    ----------
    A : np.ndarray
        Symmetric positive definite matrix of shape (n, n).
    B : np.ndarray
        Matrix of shape (n, m), each column is a right-hand side vector.

    Returns
    -------
    np.ndarray
        Solution matrix X of shape (n, m).
    """
    # Factor A once: A = L L^T (or U^T U)
    c_factor, lower = scipy.linalg.cholesky(A, lower=True, check_finite=True), True

    # Solve all RHS at once using the Cholesky factor
    X = scipy.linalg.cho_solve((c_factor, lower), B, check_finite=True)
    return X


**(c)** Time both approaches with n = 300 and m = 100, and report the speedup. Use a random symmetric positive definite matrix (for example, `A = Z @ Z.T + n * np.eye(n)` where `Z` is random).

In [12]:
import time
import numpy as np
import scipy.linalg


# Reproducible benchmark
rng = np.random.default_rng(0)
n, m = 300, 100
Z = rng.standard_normal((n, n))
A = Z @ Z.T + n * np.eye(n)   # SPD matrix
B = rng.standard_normal((n, m))

# Warm-up
_ = solve_systems_naive(A, B)
_ = solve_systems_cholesky(A, B)

reps = 3
naive_times = []
chol_times = []

for _ in range(reps):
    t0 = time.perf_counter()
    X_naive = solve_systems_naive(A, B)
    naive_times.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    X_chol = solve_systems_cholesky(A, B)
    chol_times.append(time.perf_counter() - t0)

naive_mean = np.mean(naive_times)
chol_mean = np.mean(chol_times)
speedup = naive_mean / chol_mean
max_err = np.max(np.abs(X_naive - X_chol))

print(f"naive times (s): {naive_times}")
print(f"cholesky times (s): {chol_times}")
print(f"mean naive (s): {naive_mean:.6f}")
print(f"mean cholesky (s): {chol_mean:.6f}")
print(f"speedup (naive/cholesky): {speedup:.2f}x")
print(f"max |X_naive - X_chol|: {max_err:.3e}")


naive times (s): [0.16012762500000122, 0.1753100829999994, 0.15667133300000557]
cholesky times (s): [0.006668166000004305, 0.006906208000003744, 0.012297291999999516]
mean naive (s): 0.164036
mean cholesky (s): 0.008624
speedup (naive/cholesky): 19.02x
max |X_naive - X_chol|: 1.388e-17


## Problem 5: Optimizing Pairwise Computation

The following function computes the sum of all pairwise absolute differences in an array:

$$S = \sum_{i=0}^{n-1} \sum_{j=i+1}^{n-1} |a_i - a_j|$$

In [13]:
def pairwise_abs_diff_slow(arr):
    """Compute sum of all pairwise absolute differences.

    Parameters
    ----------
    arr : list
        A list of numbers.

    Returns
    -------
    float
        Sum of |a_i - a_j| for all pairs i < j.

    Examples
    --------
    >>> pairwise_abs_diff_slow([1, 2, 4])
    6
    >>> pairwise_abs_diff_slow([2, 8, 4, 6])
    20
    """
    n = len(arr)
    total = 0
    for i in range(n):
        for j in range(i + 1, n):
            total += abs(arr[i] - arr[j])
    return total

This runs in O(n^2) time. Your task is to write a function `pairwise_abs_diff_fast` that computes the same result in O(n log n) time.

In [14]:
def pairwise_abs_diff_fast(arr):
    """Compute sum of all pairwise absolute differences in O(n log n).

    Parameters
    ----------
    arr : list
        A list of numbers.

    Returns
    -------
    float
        Sum of |a_i - a_j| for all pairs i < j.
    """
    a = sorted(arr)
    prefix_sum = 0
    total = 0

    for k, x in enumerate(a):
        # For sorted array, x >= all previous elements.
        # Contribution with previous elements:
        # (x-a0) + (x-a1) + ... + (x-a_{k-1}) = k*x - prefix_sum
        total += k * x - prefix_sum
        prefix_sum += x

    return total


Hint: consider what happens when you sort the array first. After sorting, every element `a[k]` is greater than or equal to all elements before it. Think about how many times `a[k]` is added versus subtracted across all pairs that include index k.